In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

!pwd
import os
os.chdir('/content/drive/My Drive/Colab Notebooks')
!pwd
# Get the current working directory
current_directory = os.getcwd()

# Specify the file name
file_name = "mastershifted.txt"

# Construct the full path to the file
file_path = os.path.join(current_directory, file_name)

# Open the file
try:
    with open(file_path, "r") as f:
        # Do something with the file (e.g., print its contents)
        print("Loaded successfully")
except FileNotFoundError:
    print(f"The file '{file_name}' was not found in the current directory.")
except Exception as e:
    print(f"An error occurred: {e}")

/content
/content/drive/My Drive/Colab Notebooks
Loaded successfully


In [ ]:
#Import the necessary modules
import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Reshape, LSTM, Dense, Dropout, Activation, TimeDistributed, Flatten, BatchNormalization, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

In [ ]:
#Read the data variables

colnames=( 'Bx', 'By', 'Bz', 'Vx', 'Density', 'Labels')
df = pd.read_csv('mastershifted.txt', sep='\s+', names=colnames)
dfc=df.drop(['Labels'], axis=1)

#We have 120 minute windows, each associated with either a 1 or 0
n_timesteps = 120

Xcols=[x for x in dfc.columns if x!= 'Labels']
n_features=len(Xcols)
model= Sequential()
#We use a scaler that does not standardize/normalize the values because we do not expect Gaussian data
scaler = MinMaxScaler()
X = df[Xcols].values
y=df['Labels']

#We need to use 70/15/15 for train/test/validate
#The samples are 120 rows by 6 variables. We need to specify the indices of each sample and then multiple by 120

targtrain=np.empty((53404,))
targtrain[::2]=0 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]
targtrain[1::2]=1 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]

targtest=np.empty((11444,))
targtest[::2]=0 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]
targtest[1::2]=1 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]

targvalid=np.empty((11444,))
targvalid[::2]=0 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]
targvalid[1::2]=1 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]

#The last column has 0s or 1s only for each sample (substorm or non substorm)
#These are all the row locations for each data set to be used.

X_train = X[0:6408480]
y_train = targtrain#y[0:4700760]
X_test =  X[6408480:7781760]
y_test =  targtest#y[4700760:5708040]
X_val =   X[7781760:9155040]
y_val =   targvalid#y[5708040:6715320]


In [ ]:
#We use the scaler on each set of data based on its maximum and minimum to make computation easier

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val=scaler.transform(X_val)

#Tailor the correct input format for feeding into the LSTM setup

n_samples = int(X_train.shape[0]/120)
X_tr = np.resize(X_train, (n_samples, n_timesteps, n_features))
X_te = np.resize(X_test, (11444, 120, 5))
X_va = np.resize(X_val, (11444, 120, 5))

In [ ]:
model = Sequential()

#LSTM layer with 32 units
model.add(LSTM(32, return_sequences=False))
model.add(BatchNormalization())

# Fully connected layer with 32 neurons and ReLU activation
model.add(Dense(32, activation='relu'))


#Classify
model.add(Dense(1, activation='sigmoid'))

#Compile
model.compile(optimizer=Adam(learning_rate=0.00001), loss='binary_crossentropy', metrics=['accuracy'])

#Train

history=model.fit(X_tr, y_train, epochs=45, batch_size=128, validation_data=(X_va, y_val))